# 04 · Clinical interpretation — the four analyses

**This is the contribution notebook.** Anyone can train a ResNet. The story we tell here is what makes the repo defensible:

1. **Subgroup robustness** — does AFib sensitivity hold across stratifying variables?
2. **Calibration** — are the predicted probabilities trustworthy?
3. **Failure-mode taxonomy** — when the model errs, *how* does it err?
4. **Clinician concordance** — do Grad-CAM peaks land where you'd point?

Prereq: trained checkpoint.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from af_explain.data.dataset import LABEL_NAMES, PhysioNet2017Dataset
from af_explain.evaluation.clinical_analysis import (
    RESEARCH_QUESTION,
    calibration_report,
    failure_mode_taxonomy,
    subgroup_robustness_report,
)
from af_explain.training.lightning_module import AFClassifier

DATA_ROOT = Path('../data/raw/training2017')
CKPT = Path('../outputs/checkpoints/best.ckpt')
print(RESEARCH_QUESTION)

In [ ]:
model = AFClassifier.load_from_checkpoint(str(CKPT), map_location='cpu')
model.eval()
test_ds = PhysioNet2017Dataset(DATA_ROOT, split='test', sample_mode='center')
loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

all_logits, all_labels, all_record_ids = [], [], []
with torch.no_grad():
    for batch in loader:
        all_logits.append(model(batch['signal']))
        all_labels.append(batch['label'])
        all_record_ids.extend(batch['record_id'])
logits = torch.cat(all_logits)
labels = torch.cat(all_labels)
preds = logits.argmax(1).numpy()
y = labels.numpy()
len(preds), 'records evaluated'

## 1. Subgroup robustness — record length and signal quality

We don't have ground-truth quality bins, so we proxy quality by **bandpass-filtered signal RMS** quartiles, and length by **raw record duration** quartiles.

In [ ]:
import wfdb

from af_explain.data.preprocess import bandpass_filter

rows = []
for rec in all_record_ids:
    sig, meta = wfdb.rdsamp(str(DATA_ROOT / rec))
    raw = sig[:, 0]
    rows.append({
        'record': rec,
        'length_s': raw.size / meta['fs'],
        'rms': float(np.sqrt(np.mean(bandpass_filter(raw.astype(np.float32), meta['fs']) ** 2))),
    })
meta_df = pd.DataFrame(rows)
meta_df['length_bin'] = pd.qcut(meta_df['length_s'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
meta_df['quality_bin'] = pd.qcut(meta_df['rms'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

report_len = subgroup_robustness_report(preds, y, meta_df, 'length_bin')
report_qual = subgroup_robustness_report(preds, y, meta_df, 'quality_bin')
print('--- Length subgroups ---'); print(report_len.to_markdown())
print('--- Quality subgroups ---'); print(report_qual.to_markdown())

## 2. Calibration — reliability + ECE + temperature

In [ ]:
import matplotlib.pyplot as plt

cal = calibration_report(logits, labels, n_bins=10)
print(cal.to_markdown())
print(f'Suggested temperature: T = {cal.temperature:.3f}')

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
for name, (prob_pred, prob_true) in cal.reliability_curves.items():
    ax.plot(prob_pred, prob_true, marker='o', label=name)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Empirical accuracy')
ax.set_title('Reliability diagram (one-vs-rest)')
ax.legend(); ax.set_aspect('equal'); plt.tight_layout()

## 3. Failure-mode taxonomy

In [ ]:
tax = failure_mode_taxonomy(preds, y, all_record_ids)
tax.summary()

In [ ]:
# Spot-check: pull 3 false-negative AFib records and look at them by eye.
fn_afib = tax.rows.query('failure_mode == "false_negative_afib"').head(3)
fn_afib

## 4. Clinician concordance — needs your annotations

Populate `annotations/<record_id>.json` for ~30 records spanning all classes, then run:

In [ ]:
from af_explain.evaluation.clinical_analysis import clinician_concordance
from af_explain.explain.gradcam import make_gradcam

annotated = [rec for rec in all_record_ids if (Path('../annotations') / f'{rec}.json').exists()]
if not annotated:
    print('No annotations yet. See annotations/README.md.')
else:
    saliency_per_record = {}
    cam = make_gradcam(model.model)
    for rec in annotated:
        idx = all_record_ids.index(rec)
        x = test_ds[idx]['signal'].unsqueeze(0)
        saliency_per_record[rec] = cam(x)
    cam.remove_hooks()
    result = clinician_concordance(saliency_per_record, '../annotations')
    print(f'IoU = {result.iou:.3f} · peak-in-ROI = {result.peak_inside_roi_rate:.3f} (n={result.n_records})')

## Where the results go

- Numbers feed the **Results** section of `docs/methods.qmd`.
- The Subgroup table + Failure-mode summary are screenshot-worthy for the README.
- Surprising calibration ECE values → discussion point about temperature scaling.
- High clinician concordance → core claim of the paper draft.